# MagicBrush → mixed-modality prompts

Loads the official `osunlp/MagicBrush` Hugging Face dataset with its published schema. The notebook defaults to the public `dev` split; the authors distribute the test set separately. Set `RESUME_LAST_RUN = True` in the setup cell to continue the latest annotation run, skipping completed samples and retrying audited failures; otherwise a new timestamped run folder is created.

Install once from the repository root: `uv sync`. Copy `.env.example` to `.env` and set `GEMINI_API_KEY`. Use `MAGICBRUSH_SPLIT=train` only if you intentionally want the much larger training split.

In [1]:
import os

from annotation import (
    AnnotationConfig,
    create_annotation_run_directory,
    magicbrush,
    resolve_annotation_run_directory,
    run_hf_annotation,
)

DATASET_ID = "osunlp/MagicBrush"
SPLIT = os.environ.get("MAGICBRUSH_SPLIT") or "dev"
if SPLIT not in {"train", "dev"}:
    raise ValueError("HF MagicBrush exposes train/dev; official test is distributed separately by the authors.")
CONFIG = AnnotationConfig.from_env("magicbrush")
RESUME_LAST_RUN = False  # True continues the latest run and skips completed samples.

In [2]:
from dataclasses import replace

RUN_DIR = (
    resolve_annotation_run_directory(CONFIG.output_dir)
    if RESUME_LAST_RUN
    else create_annotation_run_directory(CONFIG.output_dir)
)
RUN_CONFIG = replace(CONFIG, output_dir=RUN_DIR)
print("Annotation run directory:", RUN_DIR)

summary = run_hf_annotation(
    dataset_id=DATASET_ID,
    split=SPLIT,
    adapter=magicbrush,
    config=RUN_CONFIG,
    output_name=f"magicbrush_{SPLIT}_mixed.jsonl",
    audit_name=f"magicbrush_{SPLIT}_audit.jsonl",
    expected_columns={"img_id", "turn_index", "source_img", "mask_img", "instruction", "target_img"},
    use_grounder=True,
)
summary

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


dev: 0it [00:00, ?it/s]

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

KeyboardInterrupt: 

For a single visible target, the human-provided `mask_img` is reused as the reference region after nearest-neighbor resizing to the source-image size. Other independent references fall back to Grounding DINO.